Evaluate the resume

In [2]:
#Importing the required libraries
import os
import pickle
import json
import numpy as np
import pandas as pd
import faiss

In [4]:
#Load Saved Pickle Files
#Load Job Description
with open("job_desc.pkl", "rb") as f:
    job_data = pickle.load(f)
#Load Resume
with open("resume_metadata.pkl", "rb") as f:
    resume_data = pickle.load(f)

print("Job records:", len(job_data))
print("Resume records:", len(resume_data))

Job records: 15
Resume records: 20


In [5]:
#Function to return Job ID based on Job title
def get_job_id_by_title(job_title, job_data):
    for job in job_data:
        if job.get("title", "").strip().lower() == job_title.strip().lower():
            return job.get("job_id")

    return None

In [6]:
#test get_job_id
job_id_test = get_job_id_by_title("Data Scientist", job_data)
print(job_id_test)

3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f


UUID: 3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f for "Data Scientist"

In [7]:
#Save embeddings
#Job Embeddings FAISS
# Load FAISS index
job_index = faiss.read_index("jobs.faiss")

In [8]:
#Resume Embeddings FAISS
resume_index = faiss.read_index("resume.faiss")

In [9]:
#Function to map a FAISS result back to its original chunk
def get_job_chunk(
    index_position
):

    return job_data[
        index_position
    ]

In [10]:
#Function to map resume FAISS result back to its original chunk
def get_resume_chunk(
    index_position
):

    return resume_data[
        index_position
    ]

In [11]:
#Retrive Job Description chunks
def get_job_context(
    job_id
):

    selected_chunks = [
        item
        for item in job_data
        if item["job_id"] == job_id
    ]

    selected_chunks = sorted(
        selected_chunks,
        key=lambda x: x["chunk_id"]
    )

    context = "\n\n".join(
        item["text"]
        for item in selected_chunks
    )

    return context

In [12]:
#Retrive Resume Chunks
def get_resume_context(
    unique_id
):

    selected_chunks = [
        item
        for item in resume_data
        if item["unique_id"] == unique_id
    ]

    selected_chunks = sorted(
        selected_chunks,
        key=lambda x: x["chunk_id"]
    )

    context = "\n\n".join(
        item["text"]
        for item in selected_chunks
    )

    return context

In [13]:
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [14]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

In [15]:
from langchain_ollama import ChatOllama

In [ ]:
#Load LLM

llm = ChatOllama(
    model="gemma4:e2b",
    temperature=0
)

#load_dotenv()
#llm = ChatOpenAI(
#    model="gpt-5.6-luna",
#    temperature=0
#)

#os.environ["HUGGINGFACEHUB_API_TOKEN"] = "enter token here"

#Initialize the Hugging Face base endpoint
#llm_endpoint = HuggingFaceEndpoint(
#    repo_id="meta-llama/Llama-3.1-8B-Instruct",
#    task="text-generation",
#    max_new_tokens=1024,
#    temperature=0.0,  
#    do_sample=False
#)

#Wrap it with ChatHuggingFace to handle system/human message templates
#llm = ChatHuggingFace(llm=llm_endpoint)

In [17]:
#Test LLM
response_test = llm.invoke("Answer in not more than 100 words. can you evaluate resume and return json output?")

print(response_test.content)

Yes, I can evaluate a resume and return the feedback in JSON format.

Please provide the resume you would like me to evaluate. I will analyze its content, structure, grammar, and overall effectiveness based on standard hiring practices.


In [18]:
from langchain_core.prompts import ChatPromptTemplate

In [19]:
#Evaluation Prompt
evaluation_prompt = ChatPromptTemplate.from_template(
"""
You are an evidence-based resume evaluation assistant.

Evaluate the RESUME against the JOB DESCRIPTION.

=============================
STRICT RAG RULES
=============================

1. Use ONLY the supplied Job Description and Resume.

2. Do NOT use outside knowledge.

3. Do NOT invent:
   - skills
   - experience
   - education
   - certifications
   - projects
   - achievements
   - technologies

4. A skill can be considered matching only if the
   resume provides evidence for it.

5. A skill can be considered missing only if it is
   required by the Job Description and there is no
   supporting evidence in the Resume.

6. If information is not available in the resume,
   say:

   "Not found in the provided resume."

7. All conclusions must be based on the supplied
   documents.

=============================
JOB DESCRIPTION
=============================

{job_context}


=============================
RESUME
=============================

{resume_context}


=============================
TASK
=============================

Evaluate this resume against this job description.

Return ONLY valid JSON.

Use this exact structure:

{{
    "match_score": 0,

    "matching_skills": [],

    "missing_skills": [],

    "candidate_summary": "",

    "strengths": [],

    "weaknesses": [],

    "hiring_recommendation": "",

    "justification": ""
}}


=============================
MATCH SCORE
=============================

Provide a score from 0 to 100.

The score should reflect the evidence-based match
between the requirements in the Job Description and
the qualifications demonstrated in the Resume.

Consider:

- Required skills
- Required technologies
- Required experience
- Responsibilities
- Education
- Certifications
- Other explicit requirements

Do not award credit for information that is not
demonstrated in the resume.


=============================
MATCHING SKILLS
=============================

List the important skills required by the Job
Description that are explicitly supported by the
Resume.


=============================
MISSING SKILLS
=============================

List important requirements from the Job Description
that are not demonstrated in the Resume.


=============================
CANDIDATE SUMMARY
=============================

Provide a concise summary of the candidate based
ONLY on the supplied Resume.


=============================
STRENGTHS
=============================

Identify strengths of the candidate specifically
in relation to the Job Description.

Every strength must be supported by the Resume.


=============================
WEAKNESSES
=============================

Identify important gaps between the Job Description
and Resume.


=============================
HIRING RECOMMENDATION
=============================

Use exactly one:

"Strong Match"

"Partial Match"

"Weak Match"


=============================
JUSTIFICATION
=============================

This field is REQUIRED.

Provide a detailed 2-4 paragraph explanation.

The justification must explain:

1. Why the candidate received the match score.

2. Which important job requirements are demonstrated
   by the resume.

3. Which important requirements are not demonstrated.

4. How the candidate's experience relates to the
   responsibilities in the Job Description.

5. Why the hiring recommendation follows from the
   evidence.

6. Any limitations caused by missing information
   in the Resume.

The justification must be based ONLY on the supplied
Job Description and Resume. Restrict to 200 words.

Do not introduce external information.

Do not simply repeat the candidate summary.

Return valid JSON only.
"""
)

In [20]:
from langchain_core.output_parsers import JsonOutputParser


In [21]:
#Creating Lang Chain
evaluation_chain = (
    evaluation_prompt
    | llm
    | JsonOutputParser()
)

In [ ]:
#Evaluate 1 Resume
def evaluate_resume(
    job_id,
    unique_id
):

    # Get JD context

    job_context = get_job_context(
        job_id
    )


    # Get Resume context

    resume_context = get_resume_context(
        unique_id
    )

    # Send to LLM

    result = evaluation_chain.invoke(
        {
            "job_context":
                job_context,

            "resume_context":
                resume_context
        }
    )


    return result

In [24]:
#Fetch resume unique ID based on Job ID and Resume Name
def get_unique_id_by_job_and_resume(job_id, resume_file, resume_data):
    for resume in resume_data:
        if (
            resume.get("job_id") == job_id
            and resume.get("resume_file", "").strip().lower() == resume_file.strip().lower()
        ):
            return resume.get("unique_id")

    return None


In [26]:
#Test 
unique_id_test = get_unique_id_by_job_and_resume(
    job_id_test,
    "Resume A - Rahul Sharma.pdf",
    resume_data
)

print(unique_id_test)

3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_d4c2dc9e-fa73-4acf-b4d7-0d63a9095515


"Resume A - Rahul Sharma.pdf" --> 3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_d4c2dc9e-fa73-4acf-b4d7-0d63a9095515

In [34]:
#Example 1: Evaluate A resume against JD
result_a = evaluate_resume(
    job_id="3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f",
    unique_id="3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_d4c2dc9e-fa73-4acf-b4d7-0d63a9095515"
)

print(
    json.dumps(
        result_a,
        indent=4
    )
)

{
    "match_score": 85,
    "matching_skills": [
        "Python",
        "SQL",
        "Pandas",
        "NumPy",
        "Exploratory Data Analysis",
        "Feature Engineering",
        "Regression",
        "Classification",
        "Random Forest",
        "Gradient Boosting",
        "XGBoost",
        "Scikit-learn",
        "Power BI",
        "PostgreSQL",
        "MySQL",
        "AWS"
    ],
    "missing_skills": [
        "MLflow",
        "Docker"
    ],
    "candidate_summary": "Rahul Sharma is a Data Scientist with 4 years of experience, proficient in Python, SQL, and machine learning techniques. He has demonstrated experience in developing classification and regression models using algorithms like Random Forest, Gradient Boosting, and XGBoost, along with strong skills in feature engineering, exploratory data analysis, and statistical testing. He has experience with data visualization tools like Power BI and has deployed machine learning models on AWS.",
    "streng

In [35]:
#Resume B Unique ID
unique_id_test = get_unique_id_by_job_and_resume(
    job_id_test,
    "Resume B - Priya Mehta.pdf",
    resume_data
)

print(unique_id_test)

3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_111ff27c-93e4-4326-b3ad-b1b7da367c2f


In [36]:
#Example 1: Evaluate A resume against JD
result_b = evaluate_resume(
    job_id_test,
    "3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_111ff27c-93e4-4326-b3ad-b1b7da367c2f"
)

In [37]:
#Example 2:	Compare Resume A and Resume B for the same JD
#Display comparison result in tabular form
comparison = pd.DataFrame(
    [
        {
            "Resume":
                "Resume A",

            "Match Score":
                result_a[
                    "match_score"
                ],

            "Recommendation":
                result_a[
                    "hiring_recommendation"
                ]
        },

        {
            "Resume":
                "Resume B",

            "Match Score":
                result_b[
                    "match_score"
                ],

            "Recommendation":
                result_b[
                    "hiring_recommendation"
                ]
        }
    ]
)

comparison

,Resume,Match Score,Recommendation
0,Resume A,85,Strong Match
1,Resume B,75,Partial Match


In [38]:
#Print Justification of 2 resumes
print(
    "Resume A justification:"
)

print(
    result_a["justification"]
)


print(
    "\nResume B justification:"
)

print(
    result_b["justification"]
)

Resume A justification:
The candidate receives a strong match score of 85 due to extensive overlap with the core technical requirements of the Data Scientist role. Rahul Sharma possesses the required 4 years of experience and strong proficiency in essential skills such as Python, SQL, Pandas, NumPy, and statistical analysis. The resume explicitly demonstrates experience in developing classification and regression models using algorithms like Random Forest, Gradient Boosting, and XGBoost, fulfilling key machine learning requirements. Furthermore, the candidate has experience with feature engineering, model evaluation metrics (accuracy, precision, recall, F1-score, ROC-AUC), and data visualization (Power BI), directly addressing the key responsibilities of the job description.

The candidate successfully demonstrates the core analytical workflow, from data collection and cleaning to model deployment on AWS. The experience aligns well with the responsibilities of collecting, transforming,

In [39]:
#Resume C Unique ID
unique_id_test = get_unique_id_by_job_and_resume(
    job_id_test,
    "Resume C - Amit Verma.pdf",
    resume_data
)

print(unique_id_test)

3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_11911258-3cc6-4598-ba42-a156575796cf


In [40]:
#Evaluate result C
result_c = evaluate_resume(
    job_id=job_id_test,
    unique_id="3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_11911258-3cc6-4598-ba42-a156575796cf"
)

In [41]:
#Print Missing Skills in Resume C
print(
    "Missing Skills:"
)

for skill in result_c[
    "missing_skills"
]:

    print(
        "❌",
        skill
    )

print(
    "\nJustification:"
)

print(
    result_c[
        "justification"
    ]
)

Missing Skills:
❌ Strong proficiency in Python
❌ Supervised and unsupervised machine learning
❌ Regression and classification algorithms
❌ Random Forest
❌ Gradient Boosting
❌ XGBoost
❌ scikit-learn
❌ Feature engineering
❌ Model evaluation metrics (accuracy, precision, recall, F1-score, ROC-AUC)
❌ Experience with AWS or another cloud platform
❌ Experience deploying machine learning models
❌ Docker
❌ MLflow

Justification:
The candidate received a partial match score because while they possess foundational data analysis skills and relevant experience duration, they lack the core, demonstrated expertise required for a Data Scientist role. The resume shows experience in SQL, Pandas, and data visualization (Power BI), which aligns with some responsibilities like collecting, cleaning, and visualizing data. However, the resume does not provide evidence of the strong proficiency in Python, advanced machine learning algorithms (like Random Forest, XGBoost, scikit-learn), or the full machine lea

In [42]:
#Compare all resume
def get_resumes_for_job(
    job_id
):

    resume_ids = list(
        dict.fromkeys(
            item["unique_id"]
            for item in resume_data
            if item["job_id"] == job_id
        )
    )

    return resume_ids

In [43]:
job_id = job_id_test

resume_ids = get_resumes_for_job(
    job_id
)

print(
    "Number of candidates:",
    len(resume_ids)
)

Number of candidates: 3


In [ ]:
#Evaluate all JD
#Example 4: Recommend the best candidate among multiple resumes
all_results = []

for resume_id in resume_ids:

    result = evaluate_resume(
        job_id,
        resume_id
    )

    all_results.append(
        {
            "resume_id":
                resume_id,

            "match_score":
                result[
                    "match_score"
                ],

            "matching_skills":
                result[
                    "matching_skills"
                ],

            "missing_skills":
                result[
                    "missing_skills"
                ],

            "recommendation":
                result[
                    "hiring_recommendation"
                ],

            "candidate_summary":
                result[
                    "candidate_summary"
                ],

            "strengths":
                result[
                    "strengths"
                ],

            "weaknesses":
                result[
                    "weaknesses"
                ],

            "justification":
                result[
                    "justification"
                ]
        }
    )


In [ ]:
#Dataframe to display the comparison result
#Example 5: Generate a hiring recommendation with justification
results_df = pd.DataFrame(
    all_results
)

results_df

,resume_id,match_score,matching_skills,missing_skills,recommendation,candidate_summary,strengths,weaknesses,justification
0,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_d4c2dc9e-...,85,"[Python, SQL, Pandas, NumPy, Exploratory Data ...","[MLflow, Docker]",Strong Match,Rahul Sharma is a Data Scientist with 4 years ...,[Strong proficiency in core required skills: P...,[The resume does not explicitly demonstrate ex...,The candidate receives a strong match score of...
1,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_111ff27c-...,75,"[Python, SQL, Pandas, NumPy, Scikit-learn, Ran...","[Experience with XGBoost, Experience with AWS ...",Partial Match,Priya Mehta is a Data professional with 5 year...,"[Strong proficiency in Python, Pandas, and Num...",[Limited experience with relational database d...,The candidate demonstrates a strong foundation...
2,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_11911258-...,45,"[SQL, Pandas, Power BI, Descriptive Statistics...","[Strong proficiency in Python, Supervised and ...",Partial Match,The candidate is a Data Analyst with 3 years o...,"[Experience with SQL and MySQL., Experience wi...",[The resume does not demonstrate strong profic...,The candidate received a partial match score b...


In [46]:
job_result_df = results_df.copy()

job_result_df.insert(
    0,
    "job_id",
    job_id
)

In [47]:
job_result_df

,job_id,resume_id,match_score,matching_skills,missing_skills,recommendation,candidate_summary,strengths,weaknesses,justification
0,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_d4c2dc9e-...,85,"[Python, SQL, Pandas, NumPy, Exploratory Data ...","[MLflow, Docker]",Strong Match,Rahul Sharma is a Data Scientist with 4 years ...,[Strong proficiency in core required skills: P...,[The resume does not explicitly demonstrate ex...,The candidate receives a strong match score of...
1,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_111ff27c-...,75,"[Python, SQL, Pandas, NumPy, Scikit-learn, Ran...","[Experience with XGBoost, Experience with AWS ...",Partial Match,Priya Mehta is a Data professional with 5 year...,"[Strong proficiency in Python, Pandas, and Num...",[Limited experience with relational database d...,The candidate demonstrates a strong foundation...
2,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f,3d700044-ee52-4e1a-98c9-1c3a4fb9ac1f_11911258-...,45,"[SQL, Pandas, Power BI, Descriptive Statistics...","[Strong proficiency in Python, Supervised and ...",Partial Match,The candidate is a Data Analyst with 3 years o...,"[Experience with SQL and MySQL., Experience wi...",[The resume does not demonstrate strong profic...,The candidate received a partial match score b...


In [48]:
#Save job results in pickle file
with open("job_result.pkl", "wb") as f:
    pickle.dump(job_result_df, f)